## メッセージのトリミング

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model, after_model
from langgraph.runtime import Runtime
from langchain_core.runnables import RunnableConfig
from typing import Any
from dotenv import load_dotenv
import os

# .env ファイルから環境変数を読み込む
load_dotenv(override=True)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

# 要約生成用のモデルを初期化
model = init_chat_model(
    model="openai/gpt-5.4-mini",
    model_provider="openai",
    # profile={"max_input_tokens": 128_000},
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)


In [ ]:
# ミドルウェアでメッセージをトリミングする
@before_model
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    messages = state["messages"]
    if len(messages) <= 3:
        return None
    first_msg = messages[0]
    # 偶数件のデータがある場合は直近の3件、奇数件の場合は直近の4件のメッセージを取得する
    recent_messages = messages[-3:] if len(messages) % 2 == 0 else messages[-4:]
    new_messages = [first_msg] + recent_messages

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *new_messages
        ]
    }


agent = create_agent(
    model=model,
    middleware=[trim_messages],
    checkpointer=InMemorySaver(),
)
config: RunnableConfig = {"configurable": {"thread_id": "1"}}
agent.invoke({"messages": [HumanMessage("こんにちは、私は田中です")]}, config)
agent.invoke({"messages": [HumanMessage("これからあなたを鈴木と呼びます")]}, config)
agent.invoke({"messages": [HumanMessage("今日はいい天気ですね")]}, config)
final_response = agent.invoke({"messages": [HumanMessage("教えてください、あなたは誰ですか？私は誰ですか？")]}, config)
for msg in final_response["messages"]:
    msg.pretty_print()


## メッセージの削除

In [ ]:
# ミドルウェアでメッセージをトリミングする
@after_model
def delete_old_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    messages = state["messages"]
    if len(messages) > 5:
        to_delete = len(messages) - 5

        return {
            "messages": [
                RemoveMessage(id=m.id) for m in messages[0:to_delete]
            ]
        }
    return None


agent = create_agent(
    model=model,
    middleware=[delete_old_messages],
    checkpointer=InMemorySaver(),
)
config: RunnableConfig = {"configurable": {"thread_id": "1"}}
agent.invoke({"messages": [HumanMessage("こんにちは、私は田中です")]}, config)
agent.invoke({"messages": [HumanMessage("これからあなたを鈴木と呼びます")]}, config)
agent.invoke({"messages": [HumanMessage("今日はいい天気ですね")]}, config)
final_response = agent.invoke({"messages": [HumanMessage("教えてください、あなたは誰ですか？私は誰ですか？")]}, config)
for msg in final_response["messages"]:
    msg.pretty_print()


## 要約

In [ ]:
model_out = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_API_BASE")
)
model_in = init_chat_model(
    model="gpt-4o-mini",
    model_provider="openai",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_API_BASE")
)

In [ ]:
from langchain.agents.middleware import SummarizationMiddleware

# 要約ミドルウェア付きの Agent を作成
agent = create_agent(
    model=model_out,
    tools=[],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model_in,
            trigger=[
                ("tokens", 100),  # 100 トークンを超えたら要約する
            ],
            keep=("messages", 2),
            summary_prompt="過去のメッセージを要約してください。メッセージ一覧は以下の通りです\n{messages}",
        )
    ]
)

config = {"configurable": {"thread_id": "1"}}
print("\n複数ターンの対話を実行...")
conversations = [
    "私は田中と申します、エンジニアです。ここから非常に長い、非常に長い無駄話が続きます..." * 20,  # 強制的に 100 トークンを超えさせる
    "私の情報を要約してください"
]
for msg in conversations:
    response = agent.invoke(
        {"messages": [{"role": "user", "content": msg}]},
        config=config
    )
    for msg in response["messages"]:
        msg.pretty_print()
        print("*" * 50)

In [ ]:
from rich import print
final_state = agent.get_state(config=config)
print(final_state)